# Supabase 원천 1분봉 연결 및 프로파일링

Supabase PostgreSQL의 `public.candles`를 제한된 범위로 조회하고,
Iceberg Bronze 스키마로 옮기기 전에 컬럼과 품질을 확인하는 실험입니다.

**상태:** 원천 연결·프로파일링까지만 수행하며 Iceberg에는 쓰지 않습니다.

**보여주는 역량:** 외부 PostgreSQL 연결, 비밀정보 분리, 제한된 범위 조회,
원천→Bronze 스키마 매핑

## Goal

- 접속 정보는 환경변수로만 받습니다.
- 최신 100행만 조회해 원천 스키마와 결측치를 점검합니다.
- 실제 이관 전에 명시적인 Bronze 컬럼 매핑을 확인합니다.

## Setup

필요한 패키지:

```bash
pip install pandas sqlalchemy psycopg2-binary
```

필요한 환경변수:

```text
SUPABASE_DB_HOST
SUPABASE_DB_USER
SUPABASE_DB_PASSWORD
SUPABASE_DB_PORT       # 선택, 기본 5432
SUPABASE_DB_NAME       # 선택, 기본 postgres
```

비밀번호는 노트북 코드나 출력에 저장하지 않습니다.

In [ ]:
import os

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

SAMPLE_ROWS = 100
REQUIRED_ENV = (
    "SUPABASE_DB_HOST",
    "SUPABASE_DB_USER",
    "SUPABASE_DB_PASSWORD",
)

missing_env = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing_env:
    raise RuntimeError(
        f"필수 환경변수를 설정하세요: {', '.join(missing_env)}"
    )

## Steps

### 1. 최신 원천 데이터 조회

In [ ]:
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["SUPABASE_DB_USER"],
    password=os.environ["SUPABASE_DB_PASSWORD"],
    host=os.environ["SUPABASE_DB_HOST"],
    port=int(os.getenv("SUPABASE_DB_PORT", "5432")),
    database=os.getenv("SUPABASE_DB_NAME", "postgres"),
)

source_query = text("""
    SELECT
        source,
        symbol,
        interval,
        candle_time,
        open_price,
        high_price,
        low_price,
        close_price,
        volume,
        ingest_time
    FROM public.candles
    ORDER BY candle_time DESC
    LIMIT :sample_rows
""")

engine = create_engine(db_url, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(
            source_query,
            connection,
            params={"sample_rows": SAMPLE_ROWS},
        )
finally:
    engine.dispose()

print(f"조회 결과: {source_df.shape[0]}행 × {source_df.shape[1]}열")
display(source_df.head(10))

### 2. 원천 데이터 프로파일링

In [ ]:
profile_df = pd.DataFrame({
    "dtype": source_df.dtypes.astype(str),
    "null_count": source_df.isna().sum(),
    "distinct_count": source_df.nunique(dropna=True),
})
display(profile_df)

### 3. Bronze 스키마으로 매핑

| Supabase | Bronze Iceberg | 설명 |
|---|---|---|
| `open_price` | `open` | 시가 |
| `high_price` | `high` | 고가 |
| `low_price` | `low` | 저가 |
| `close_price` | `close` | 종가 |
| `ingest_time` | `collected_at` | 수집 시각 |

In [ ]:
bronze_df = source_df.rename(columns={
    "open_price": "open",
    "high_price": "high",
    "low_price": "low",
    "close_price": "close",
    "ingest_time": "collected_at",
})

expected_columns = {
    "source",
    "symbol",
    "interval",
    "candle_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "collected_at",
}
missing_columns = expected_columns - set(bronze_df.columns)
assert not missing_columns, sorted(missing_columns)

display(bronze_df.head(10))

## Checks

In [ ]:
invalid_ohlc = bronze_df[
    (bronze_df["high"] < bronze_df[["open", "close", "low"]].max(axis=1))
    | (bronze_df["low"] > bronze_df[["open", "close", "high"]].min(axis=1))
]

print(f"중복 행 수: {bronze_df.duplicated().sum()}")
print(f"OHLC 규칙 위반 행 수: {len(invalid_ohlc)}")

## Next Steps

- 수집 실행마다 `run_id`를 부여합니다.
- Pandas→Spark 스키마 변환을 명시적으로 정의합니다.
- 검증을 통과한 데이터만 `lakehouse.bronze.minute_prices_raw`에 append합니다.
- 운영 환경에서는 JDBC 기반 분할 읽기와 증분 기준을 추가합니다.